In [5]:
!pip install flask flask-cors transformers torch pyngrok accelerate -q
!ngrok authtoken [YOUR_NGROK_TOKEN]

from transformers import AutoModelForCausalLM, AutoTokenizer
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

print(f"Loading {MODEL_NAME} model... This is a high-quality model, please wait.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully on {model.device}!")

def predict_next_words(text, num_predictions=5):
    if not text:
        return []

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            num_return_sequences=num_predictions * 2,
            do_sample=True,
            temperature=0.4,
            top_k=40,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )

    predicted_words = []
    for output in outputs:
        new_tokens = output[input_length:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

        parts = generated_text.split()
        if not parts:
            continue

        first_word = parts[0]
        first_word = first_word.rstrip('.,!?;:"\'-)(')

        if first_word and first_word[0].isalpha():
            if first_word.lower() not in [w.lower() for w in predicted_words]:
                predicted_words.append(first_word)
                if len(predicted_words) >= num_predictions:
                    break

    return predicted_words

app = Flask(__name__)
CORS(app, origins=["*"])

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({"status": "healthy"})

@app.route('/api/predict', methods=['POST'])
def predict():
    data = request.get_json()
    text = data.get('text', '')
    count = data.get('count', 5)

    predictions = predict_next_words(text, count)
    return jsonify({"success": True, "predictions": predictions, "input_text": text})

public_url = ngrok.connect(5002)
print(f"\n API URL: {public_url}\n")
app.run(port=5002)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Loading Qwen/Qwen2.5-1.5B model... This is a high-quality model, please wait.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully on cuda:0!

🚀 API URL: NgrokTunnel: "https://excursional-elease-undistrustfully.ngrok-free.dev" -> "http://localhost:5002"

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5002
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:23:54] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:23:55] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:23:56] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:00] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:02] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:06] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:09] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:10] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Feb/2026 19:24:13] "POST /api/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.